# Quant Rush · Workshop 2: Reading the Vein

Last week you made a first submission with a one-line rule. Before building models, a good prospector reads the ground. This notebook does three things:

1. **Look at the data:** its shape, the columns, and the target.
2. **Understand the metric:** what it rewards, what it ignores, and why scores look small.
3. **Hunt for signal:** measure how every column relates to the target, and whether that holds over time.

It ends with exercises and a submission template. It runs in about a minute.

**How to run it.** Either:

- **On Kaggle:** create a new notebook, choose **File → Import Notebook** and upload this file, then use **+ Add Input** to attach the Quant Rush competition data.
- **On your computer:** download the competition files from the competition's Data tab, and put `train.parquet` and `test.parquet` in the same folder as this notebook (or a folder inside it). You need Python with pandas, pyarrow, and matplotlib.

## 1. Load the data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Find the folder with the competition files: on Kaggle it is under /kaggle/input.
DATA = next(p.parent for p in [*Path("/kaggle/input").rglob("train.parquet"), *Path(".").rglob("train.parquet")])

train = pd.read_parquet(DATA / "train.parquet")
test = pd.read_parquet(DATA / "test.parquet")
INPUTS = [c for c in train.columns if c not in ("id", "era", "target")]
print("train:", train.shape, "  test:", test.shape, "  input columns:", len(INPUTS))

## 2. The shape of the data

Each row is one asset in one **era** (a moment in time). Train eras are numbered in time order, so era 0 is the oldest. Test eras carry random labels.

In [ ]:
per_era = train.groupby("era").size()
print("train eras:", train["era"].nunique(), "  test eras:", test["era"].nunique())
print("assets per era: min", per_era.min(), " median", int(per_era.median()), " max", per_era.max())
print("missing values: train", int(train[INPUTS].isna().sum().sum()), "  test", int(test[INPUTS].isna().sum().sum()))

## 3. The columns

There are four **channels** for each asset, each with 12 hourly values (`00` is the latest hour, `11` the oldest), plus two **market** columns per hour:

In [ ]:
groups = {}
for c in INPUTS:
    groups.setdefault(c.rsplit("_", 1)[0], []).append(c)
pd.DataFrame({"columns": [f"{v[0]} … {v[-1]}" for v in groups.values()], "count": [len(v) for v in groups.values()]},
             index=list(groups))

The asset channels are **standardized within each era**: they say how an asset compares with the other assets at that moment, not with its own past. Let's check what that looks like for the latest hour.

In [ ]:
latest = ["ret_00", "vol_00", "range_00", "taker_00"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, c in zip(axes, latest):
    ax.hist(train[c], bins=100)
    ax.set_title(c)
plt.tight_layout()
plt.show()

train[latest].describe().round(2)

Things to notice:

- Each column is centered around 0, and most values sit between −2 and +2.
- `ret_00` and `taker_00` stop at exactly −5 and +5: extreme values are **clipped**, so one wild asset cannot dominate.
- `vol_k` and `range_k` are measured against the asset's own 12-hour average, so each row's 12 values average to about 0:

In [ ]:
vol_cols = [f"vol_{k:02d}" for k in range(12)]
print("largest row average of vol_00 … vol_11:", train[vol_cols].mean(axis=1).abs().max().round(3))

The market columns are the **same on every row of an era**. They describe the whole market that hour: `mkt_ret_k` is the median return and `mkt_disp_k` how spread out returns were. Together they turn the standardized `ret_k` back into an approximate raw return:

`raw return ≈ mkt_ret_k + ret_k × mkt_disp_k`

In [ ]:
print("distinct mkt_ret_00 values within an era:", train.groupby("era")["mkt_ret_00"].nunique().max())
raw_ret_00 = train["mkt_ret_00"] + train["ret_00"] * train["mkt_disp_00"]
raw_ret_00.describe().round(4)

## 4. The target

The target is how the asset did over the next 4 hours, **risk-adjusted**: its return divided by its recent volatility. A 3% move means more for a calm asset than for a wild one.

It is released as a **rank within the era**, scaled to −0.5 (worst) … +0.5 (best). So in every era the target is spread evenly, and its histogram is flat:

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(train["target"], bins=50)
plt.title("target")
plt.show()

one_era = train[train["era"] == train["era"].max()]["target"].sort_values()
print("one era, lowest and highest targets:", one_era.head(3).round(4).tolist(), "…", one_era.tail(3).round(4).tolist())

## 5. The metric

Kaggle scores a submission with the **mean per-era Spearman correlation**: within each era, how well do your predictions put the assets in order? Then the average over all eras.

Here is the same metric, written to run fast on many columns at once. `per_era_corr` returns one row per era and one column per input.

In [ ]:
def per_era_corr(df, cols, target="target"):
    """Spearman correlation of each column with the target, era by era. Constant columns score 0 in that era."""
    era = df["era"]
    ranks = df[cols + [target]].groupby(era).rank().astype("float32")
    centered = ranks - ranks.groupby(era).transform("mean")
    y = centered[target]
    num = centered[cols].mul(y, axis=0).groupby(era).sum()
    den = np.sqrt((centered[cols] ** 2).groupby(era).sum().mul((y ** 2).groupby(era).sum(), axis=0))
    return (num / den.where(den > 0)).fillna(0.0)


def era_score(df, prediction="prediction"):
    """The competition metric: the mean over eras of the Spearman correlation between prediction and target."""
    return per_era_corr(df, [prediction])[prediction].mean()


recent = train[train["era"] > train["era"].max() - 1080].copy()  # roughly the last six months of train
recent["prediction"] = -recent["ret_00"]
print("reversal rule on recent eras:", round(era_score(recent), 4))

### What the metric ignores

Only the **order within each era** counts. Watch what happens when we change the predictions without changing that order:

In [ ]:
p = -recent["ret_00"]
shift = recent["era"].map(pd.Series(np.random.default_rng(0).normal(size=recent["era"].nunique()),
                                    index=recent["era"].unique()))
variants = {
    "-ret_00": p,
    "× 100": p * 100,
    "cubed": p ** 3,
    "+ a different constant per era": p + shift,
    "the same number for every asset": p * 0 + 1,
    "sign flipped": -p,
}
pd.Series({name: round(era_score(recent.assign(prediction=v)), 4) for name, v in variants.items()}, name="score")

So:

- **Scale and shape don't matter.** Multiplying, cubing, or any transform that keeps the order gives the same score.
- **Comparisons across eras don't matter.** Adding a different number to each era changes nothing. You never need to predict whether the whole market goes up.
- **A constant prediction scores 0**, and flipping the sign flips the score.

### Why scores look small

A score of 0.05 sounds tiny. Look at the score of the reversal rule in each recent era separately:

In [ ]:
recent_scores = per_era_corr(recent, ["prediction"])["prediction"]
plt.figure(figsize=(8, 3))
plt.hist(recent_scores, bins=60)
plt.axvline(0, color="black", lw=1)
plt.axvline(recent_scores.mean(), color="red", lw=2, label=f"mean {recent_scores.mean():.3f}")
plt.legend()
plt.title("reversal rule, score per era")
plt.show()

print(f"eras with a positive score: {(recent_scores > 0).mean():.0%}")
print(f"spread of per-era scores (std): {recent_scores.std():.3f}")
print(f"uncertainty of the mean (std / sqrt(eras)): ±{recent_scores.std() / np.sqrt(len(recent_scores)):.4f}")

Per-era scores swing widely, and the rule loses in many eras. What makes it a real signal is that it wins **more often than it loses, consistently**. In markets, a steady 0.05 is a good result.

The last line matters for the leaderboard too: with scores this noisy, two submissions that differ by a few thousandths may be equally good. Workshop 4 is about exactly this.

## 6. Hunting for signal

The same per-era correlation works for any column. For each input, we measure its correlation with the target in every train era, then average. Quants call this the **IC** (information coefficient). Positive means "higher value, better target"; negative means the opposite.

In [ ]:
ic = per_era_corr(train, INPUTS)
summary = pd.DataFrame({"mean IC": ic.mean(), "eras positive": (ic > 0).mean()}).round(4)

channels = ["ret", "vol", "range", "taker"]
fig, ax = plt.subplots(figsize=(10, 4))
for ch in channels:
    ax.plot(range(12), [summary.loc[f"{ch}_{k:02d}", "mean IC"] for k in range(12)], marker="o", label=ch)
ax.axhline(0, color="black", lw=1)
ax.set_xlabel("hours back (0 = latest)")
ax.set_ylabel("mean IC")
ax.set_xticks(range(12))
ax.legend()
plt.show()

summary.sort_values("mean IC").head(8)

Questions to think about:

- The **latest hour's return** has the strongest (negative) relation. How quickly does it fade as you go back in time?
- `vol` and `range` change sign: negative for recent hours, positive for old ones. Remember that each row's 12 values average to 0. What single idea could explain both ends?
- The market columns are missing from the chart: they are the same for every asset in an era, so on their own they cannot rank assets (their IC is exactly 0). They can still matter in combination with other columns, which is a topic for Workshop 3.

## 7. Does it hold over time?

Train eras are in time order, so we can watch a signal over time. Here is the reversal signal's IC in every train era, with a rolling average over 180 eras:

In [ ]:
rev = -ic["ret_00"]
plt.figure(figsize=(12, 3))
plt.plot(rev.index, rev.values, lw=0.3, alpha=0.4)
plt.plot(rev.index, rev.rolling(180).mean().values, color="red", lw=2, label="rolling mean (180 eras)")
plt.axhline(0, color="black", lw=1)
plt.xlabel("train era (oldest → newest)")
plt.legend()
plt.show()

thirds = np.array_split(rev, 3)
print("reversal IC by third of train:", [round(float(t.mean()), 4) for t in thirds])

The signal is real in every part of train, but it **weakens over time**. The test set comes after the training set, so expect your score there to be lower than on old train eras.

That is why we check ideas on the most **recent** train eras (`recent` above), not on all of train: it is the closest thing we have to the test set.

## 8. Your turn

1. **Pick a rule.** Edit `my_rule` below. Ideas: use more than one hour of returns, add another channel, or weight columns differently. Use the IC table above as a map.
2. **Check it on recent eras** with `era_score`. Compare with the reversal rule's score above.
3. **Submit it** if it beats the reversal rule. Remember: 5 submissions a day per team.

In [ ]:
def my_rule(df):
    # Start from the reversal rule and change it.
    return -df["ret_00"]


recent["prediction"] = my_rule(recent)
print("my rule on recent eras:", round(era_score(recent), 4))

In [ ]:
submission = pd.DataFrame({"id": test["id"], "prediction": my_rule(test)})
submission.to_csv("submission.csv", index=False)
submission.head()

To submit, save a version of this notebook and submit its output `submission.csv`, or download the file and upload it on the competition's **Submit Predictions** page.

## Recap

- Every input compares an asset with the other assets **in the same era**. The target and the metric do the same.
- The metric only cares about **order within each era**. Scale, shape, and market-wide moves don't matter.
- Scores are small and noisy. **Consistency** across eras is what makes a signal real.
- Signals drift. Validate on the **latest** train eras.

**Next week, Workshop 3:** turning these columns into features, and the first real models. Questions are welcome on Discord. Dig well.